<a href="https://colab.research.google.com/github/MohHaroon/XAI-based-ZSL-for-IIDS/blob/master/model_train_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train and Evaluate the models - XGBoost, CatBoost, Decision tree

### EDA & Pre-processing the SDN dataset by Kaan Sarica and Angin 2020

#### EDA

In [ ]:
import pandas as pd

In [ ]:
# 0:Normal
# 1:DoS
# 2:DDoS
# 3:Port Scanning
# 4:OS Fingerprinting
# 5:Fuzzing

# for training
SDN_df_5 = pd.read_csv("/content/drive/MyDrive/IRP/SDN-Dataset/SDN-Dataset-master/SDN-Dataset-master/5-iot/35000_each.csv")

# for evaluation
SDN_df_10 = pd.read_csv("/content/drive/MyDrive/IRP/SDN-Dataset/SDN-Dataset-master/SDN-Dataset-master/10-iot/35000_each.csv")

In [ ]:
print(f"SDN_df_5 unique Labels: {SDN_df_5.columns}")
print(f"SDN_df_10 unique Labels: {SDN_df_10.columns}")

SDN_df_5 unique Labels: Index(['srcMAC', 'dstMAC', 'srcIP', 'dstIP', 'srcPort', 'dstPort', 'last_seen',
       'Protocol', 'proto_number', 'Dur', 'Mean', 'Stddev', 'Min', 'Max',
       'Pkts', 'Bytes', 'Spkts', 'Dpkts', 'Sbytes', 'Dbytes', 'Srate', 'Drate',
       'Sum', 'TnBPSrcIP', 'TnBPDstIP', 'TnP_PSrcIP', 'TnP_PDstIP',
       'TnP_PerProto', 'TnP_Per_Dport', 'N_IN_Conn_P_DstIP',
       'N_IN_Conn_P_SrcIP', 'Attack', 'Category'],
      dtype='object')
SDN_df_10 unique Labels: Index(['srcMAC', 'dstMAC', 'srcIP', 'dstIP', 'srcPort', 'dstPort', 'last_seen',
       'Protocol', 'proto_number', 'Dur', 'Mean', 'Stddev', 'Min', 'Max',
       'Pkts', 'Bytes', 'Spkts', 'Dpkts', 'Sbytes', 'Dbytes', 'Srate', 'Drate',
       'Sum', 'TnBPSrcIP', 'TnBPDstIP', 'TnP_PSrcIP', 'TnP_PDstIP',
       'TnP_PerProto', 'TnP_Per_Dport', 'N_IN_Conn_P_DstIP',
       'N_IN_Conn_P_SrcIP', 'Attack', 'Category'],
      dtype='object')


In [ ]:
print(f"SDN_df_5 shape: {SDN_df_5.shape}")
print(f"SDN_df_10 shape: {SDN_df_10.shape}")

SDN_df_5 shape: (210000, 33)
SDN_df_10 shape: (210000, 33)


In [ ]:
for col in SDN_df_5.columns:
    if SDN_df_5[col].isnull().any():
        print(f"SDN_df_5 has missing values in column: {col}")


SDN_df_5 has missing values in column: srcPort
SDN_df_5 has missing values in column: dstPort


#### Pre-processing


In [ ]:
SDN_df_model = SDN_df_5.copy()
SDN_df_model.drop(columns=['srcMAC', 'dstMAC', 'Protocol','srcIP', 'dstIP', 'srcPort', 'dstPort', 'last_seen'], inplace=True)

In [ ]:
print(f"Columns that are not numeric: {SDN_df_model.select_dtypes(exclude=['number']).columns.tolist()}")

Columns that are not numeric: []


In [ ]:
print(f"data types:\n{SDN_df_model.dtypes}")

data types:
proto_number         float64
Dur                  float64
Mean                 float64
Stddev               float64
Min                  float64
Max                  float64
Pkts                 float64
Bytes                float64
Spkts                float64
Dpkts                float64
Sbytes               float64
Dbytes               float64
Srate                float64
Drate                float64
Sum                  float64
TnBPSrcIP            float64
TnBPDstIP            float64
TnP_PSrcIP           float64
TnP_PDstIP           float64
TnP_PerProto         float64
TnP_Per_Dport        float64
N_IN_Conn_P_DstIP    float64
N_IN_Conn_P_SrcIP    float64
Attack               float64
Category             float64
dtype: object


In [ ]:
SDN_train_test_data = SDN_df_model.copy()
SDN_train_test_data.drop(columns=['Category'], inplace=True)

SDN_train_test_data.to_csv("/content/drive/MyDrive/IRP/SDN-Dataset/SDN_data.csv", index=False)

### Load Dataset

In [1]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
SDN_train_test_data = pd.read_csv("/content/drive/MyDrive/IRP/SDN-Dataset/SDN_data.csv")

In [5]:
def scale_data(data):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data)
    return scaled_data

In [6]:
SDN_scaled_data = scale_data(SDN_train_test_data.drop(columns=['Attack']))

In [7]:
def split_data(data, labels, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=test_size, random_state=random_state)
    return X_train, X_test, y_train, y_test

In [8]:
data_train, data_test, labels_train, labels_test = split_data(SDN_scaled_data, SDN_train_test_data['Attack'])

### Model Training and Evaluation

In [10]:
!pip install xgboost
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.5 MB/s eta 0:00:00


In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import brier_score_loss
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import brier_score_loss
from catboost import CatBoostClassifier

In [12]:
def train_and_evaluate_models(model, data_train, labels_train, data_test, labels_test):
  # Training Time
  start_train = time.time()
  model.fit(data_train, labels_train)
  end_train = time.time()
  training_duration = end_train - start_train

  # Inference Time
  start_inf = time.time()
  preds = model.predict(data_test)
  end_inf = time.time()
  inference_duration = (end_inf - start_inf) / len(data_test) # Average per packet

  probs = bst.predict_proba(data_test)[:, 1]
  bs = brier_score_loss(labels_test, probs)

  return preds, training_duration, inference_duration,bs

#### XGBoost

In [13]:
bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary:logistic')

preds, training_duration_xgb, inference_duration_xgb,bs = train_and_evaluate_models(bst, data_train, labels_train, data_test, labels_test)

accuracy = accuracy_score(labels_test, preds)

print("-" * 60)
print("XGBoost Results:")
print(f"Overall Accuracy: {accuracy * 100:.2f}%")
print(f"Brier Score: {bs:.4f}")
print(f"Training Time: {training_duration_xgb:.2f} seconds")
print(f"Inference Time: {inference_duration_xgb:.2f} seconds per packet")
print("-" * 60)

print("Classification Report:")
print(classification_report(labels_test, preds))



------------------------------------------------------------
XGBoost Results:
Overall Accuracy: 95.01%
Brier Score: 0.0431
Training Time: 1.54 seconds
Inference Time: 0.00 seconds per packet
------------------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

         0.0       0.85      0.85      0.85      6950
         1.0       0.97      0.97      0.97     35050

    accuracy                           0.95     42000
   macro avg       0.91      0.91      0.91     42000
weighted avg       0.95      0.95      0.95     42000




#### RFC

In [14]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

preds, training_duration_rf, inference_duration_rf,bs = train_and_evaluate_models(rf_model, data_train, labels_train, data_test, labels_test)
accuracy = accuracy_score(labels_test, preds)

print("-" * 60)
print("RF Results:")
print(f"Overall Accuracy: {accuracy * 100:.2f}%")
print(f"Brier Score: {bs:.4f}")
print(f"Training Time: {training_duration_rf:.2f} seconds")
print(f"Inference Time: {inference_duration_rf:.2f} seconds per packet")
print("-" * 60)

print("Classification Report:")
print(classification_report(labels_test, preds))

------------------------------------------------------------
RF Results:
Overall Accuracy: 98.75%
Brier Score: 0.0431
Training Time: 70.03 seconds
Inference Time: 0.00 seconds per packet
------------------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.94      0.96      6950
         1.0       0.99      1.00      0.99     35050

    accuracy                           0.99     42000
   macro avg       0.98      0.97      0.98     42000
weighted avg       0.99      0.99      0.99     42000



#### DT

In [17]:
dt_model = DecisionTreeClassifier(random_state=42)

preds, training_duration_dt, inference_duration_dt,bs = train_and_evaluate_models(dt_model, data_train, labels_train, data_test, labels_test)
accuracy = accuracy_score(labels_test, preds)

print("-" * 60)
print("DT Results:")
print(f"Overall Accuracy: {accuracy * 100:.2f}%")
print(f"Brier Score: {bs:.4f}")
print(f"Training Time: {training_duration_dt:.2f} seconds")
print(f"Inference Time: {inference_duration_dt:.2f} seconds per packet")
print("-" * 60)

print("Classification Report:")
print(classification_report(labels_test, preds))

------------------------------------------------------------
DT Results:
Overall Accuracy: 98.59%
Brier Score: 0.0431
Training Time: 4.87 seconds
Inference Time: 0.00 seconds per packet
------------------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      0.94      0.96      6950
         1.0       0.99      0.99      0.99     35050

    accuracy                           0.99     42000
   macro avg       0.98      0.97      0.97     42000
weighted avg       0.99      0.99      0.99     42000



#### CatBoost

In [18]:
# Initialize CatBoost
# similar parameters to your XGBoost setup for a fair comparison
cat_model = CatBoostClassifier(iterations=100, depth=5, learning_rate=1, verbose=0)

preds, training_duration_dt, inference_duration_dt,bs = train_and_evaluate_models(cat_model, data_train, labels_train, data_test, labels_test)
accuracy = accuracy_score(labels_test, preds)

print("-" * 60)
print("CATBoost Results:")
print(f"Overall Accuracy: {accuracy * 100:.2f}%")
print(f"Brier Score: {bs:.4f}")
print(f"Training Time: {training_duration_dt:.2f} seconds")
print(f"Inference Time: {inference_duration_dt:.2f} seconds per packet")
print("-" * 60)

print("Classification Report:")
print(classification_report(labels_test, preds))

------------------------------------------------------------
CATBoost Results:
Overall Accuracy: 98.72%
Brier Score: 0.0431
Training Time: 8.87 seconds
Inference Time: 0.00 seconds per packet
------------------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.94      0.96      6950
         1.0       0.99      1.00      0.99     35050

    accuracy                           0.99     42000
   macro avg       0.99      0.97      0.98     42000
weighted avg       0.99      0.99      0.99     42000

